# MSI Pipeline - Script 07: Image Registration (Optional)

This notebook performs VALIS-based registration of MSI data to reference images (DAPI/H&E).

## Features
- Composite image creation (PCA, SNR-weighted, or MIP)
- VALIS registration to DAPI/H&E reference
- Coordinate transformation: (x, y) -> (x_warped, y_warped)
- Quality assessment metrics

## Input
- Clustered AnnData files
- Reference images (DAPI, H&E)

## Output
- Registered images
- Transformation matrices
- Warped coordinates

## Requirements
- VALIS (pip install valis-wsi)
- libvips

In [ ]:
import sys
from pathlib import Path
import logging
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import tifffile
from skimage import exposure

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from utils import io as msi_io

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Check VALIS installation
try:
    from valis import registration
    print("VALIS is installed")
    HAS_VALIS = True
except ImportError:
    print("VALIS is not installed. Install with: pip install valis-wsi")
    print("Note: VALIS requires libvips to be installed on your system.")
    HAS_VALIS = False

## Configuration

In [ ]:
# === CONFIGURE THESE PATHS ===

# Base data directory
BASE_DIR = Path.home() / "ext_hd_sammy"

# Input: Clustered AnnData files
ANNDATA_DIR = BASE_DIR / "projects" / "out" / "out_msi" / "glycans_h5ad_processed"

# Reference images directory
REFERENCE_DIR = BASE_DIR / "data" / "stomics" / "gene_exp"

# MSI source images
MSI_IMAGE_DIR = BASE_DIR / "data" / "msi" / "Glycan OMEtif files"

# Output directory
OUTPUT_DIR = BASE_DIR / "projects" / "out" / "out_msi" / "registration"

# Sample-to-reference mapping
# Format: {sample_id: {'reference': path_to_reference, 'chip_id': chip_id}}
SAMPLE_MAPPING = {
    # Example:
    # 'SO1': {
    #     'chip_id': 'D03453A6',
    #     'reference': REFERENCE_DIR / 'D03453A6' / '03.ssDNA_analysis' / 'ssDNA_D03453A6_regist.tif',
    # },
}

# Composite image method
COMPOSITE_METHOD = 'pca'  # Options: 'pca', 'snr', 'mip' (max intensity projection)

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"AnnData directory: {ANNDATA_DIR}")
print(f"Reference directory: {REFERENCE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Create Composite Image

In [ ]:
def create_composite_pca(adata, n_components=3):
    """Create composite image using PCA of intensity data."""
    from sklearn.decomposition import PCA
    
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    coords = adata.obsm['spatial']
    
    # Get image dimensions
    x_max = int(coords[:, 0].max()) + 1
    y_max = int(coords[:, 1].max()) + 1
    
    # PCA
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(X)
    
    # Create image
    composite = np.zeros((y_max, x_max, n_components), dtype=np.float32)
    
    for i in range(len(coords)):
        x, y = int(coords[i, 0]), int(coords[i, 1])
        composite[y, x, :] = pca_result[i, :]
    
    # Normalize to 0-255
    for c in range(n_components):
        channel = composite[:, :, c]
        channel = exposure.rescale_intensity(channel, out_range=(0, 255))
        composite[:, :, c] = channel
    
    return composite.astype(np.uint8)


def create_composite_mip(adata):
    """Create composite using max intensity projection."""
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    coords = adata.obsm['spatial']
    
    x_max = int(coords[:, 0].max()) + 1
    y_max = int(coords[:, 1].max()) + 1
    
    # Max intensity per pixel
    max_intensity = X.max(axis=1)
    
    # Create image
    composite = np.zeros((y_max, x_max), dtype=np.float32)
    
    for i in range(len(coords)):
        x, y = int(coords[i, 0]), int(coords[i, 1])
        composite[y, x] = max_intensity[i]
    
    # Normalize
    composite = exposure.rescale_intensity(composite, out_range=(0, 255))
    
    return composite.astype(np.uint8)


def create_composite_snr(adata, top_n=10):
    """Create composite using SNR-weighted top channels."""
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    coords = adata.obsm['spatial']
    
    # Calculate SNR per channel
    means = X.mean(axis=0)
    stds = X.std(axis=0) + 1e-10
    snr = means / stds
    
    # Select top channels
    top_idx = np.argsort(snr)[-top_n:]
    X_top = X[:, top_idx]
    weights = snr[top_idx]
    weights = weights / weights.sum()
    
    # Weighted sum
    weighted = np.dot(X_top, weights)
    
    # Create image
    x_max = int(coords[:, 0].max()) + 1
    y_max = int(coords[:, 1].max()) + 1
    
    composite = np.zeros((y_max, x_max), dtype=np.float32)
    
    for i in range(len(coords)):
        x, y = int(coords[i, 0]), int(coords[i, 1])
        composite[y, x] = weighted[i]
    
    # Normalize
    composite = exposure.rescale_intensity(composite, out_range=(0, 255))
    
    return composite.astype(np.uint8)

In [ ]:
# Test composite creation with first sample
sample_files = list(ANNDATA_DIR.glob("*.h5ad"))

if sample_files:
    sample_file = sample_files[0]
    sample_id = sample_file.stem
    
    print(f"Loading {sample_id}...")
    adata = ad.read_h5ad(sample_file)
    
    print(f"Creating {COMPOSITE_METHOD} composite...")
    
    if COMPOSITE_METHOD == 'pca':
        composite = create_composite_pca(adata)
    elif COMPOSITE_METHOD == 'mip':
        composite = create_composite_mip(adata)
    elif COMPOSITE_METHOD == 'snr':
        composite = create_composite_snr(adata)
    
    print(f"Composite shape: {composite.shape}")
    
    # Save composite
    composite_path = OUTPUT_DIR / f"{sample_id}_composite.tif"
    tifffile.imwrite(composite_path, composite)
    print(f"Saved to {composite_path}")

In [ ]:
# Visualize composite
if 'composite' in dir():
    fig, ax = plt.subplots(figsize=(10, 10))
    
    if composite.ndim == 3:
        ax.imshow(composite)
    else:
        ax.imshow(composite, cmap='gray')
    
    ax.set_title(f'{sample_id} - {COMPOSITE_METHOD} composite')
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{sample_id}_composite_preview.png", dpi=150, bbox_inches='tight')
    plt.show()

## VALIS Registration

In [ ]:
def run_valis_registration(msi_image_path, reference_path, output_dir):
    """Run VALIS registration between MSI and reference images."""
    if not HAS_VALIS:
        raise ImportError("VALIS is required for registration")
    
    from valis import registration
    
    # Create temp directory for VALIS
    valis_dir = output_dir / "valis_temp"
    valis_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy images to VALIS directory
    import shutil
    msi_dest = valis_dir / msi_image_path.name
    ref_dest = valis_dir / reference_path.name
    
    shutil.copy(msi_image_path, msi_dest)
    shutil.copy(reference_path, ref_dest)
    
    # Run VALIS
    registrar = registration.Valis(
        str(valis_dir),
        str(output_dir / "valis_output"),
        reference_img_f=str(ref_dest.name),
        align_to_reference=True,
    )
    
    rigid_registrar, non_rigid_registrar, error_df = registrar.register()
    
    return registrar, error_df

In [ ]:
# Run registration for configured samples
if HAS_VALIS and SAMPLE_MAPPING:
    for sample_id, mapping in SAMPLE_MAPPING.items():
        print(f"\nRegistering {sample_id}...")
        
        # Paths
        msi_composite = OUTPUT_DIR / f"{sample_id}_composite.tif"
        reference = mapping['reference']
        
        if not msi_composite.exists():
            print(f"  Composite not found: {msi_composite}")
            continue
        
        if not reference.exists():
            print(f"  Reference not found: {reference}")
            continue
        
        try:
            sample_output_dir = OUTPUT_DIR / sample_id
            registrar, error_df = run_valis_registration(
                msi_composite,
                reference,
                sample_output_dir
            )
            
            print(f"  Registration complete")
            print(f"  Error metrics:\n{error_df}")
            
        except Exception as e:
            print(f"  Error: {e}")
else:
    print("VALIS not available or no samples configured")
    print("\nTo use registration:")
    print("1. Install VALIS: pip install valis-wsi")
    print("2. Install libvips on your system")
    print("3. Configure SAMPLE_MAPPING with sample-to-reference paths")

## Transform Coordinates

In [ ]:
def transform_coordinates(adata, registrar, slide_name):
    """Transform MSI coordinates using VALIS registration."""
    from valis import registration
    
    coords = adata.obsm['spatial'].copy()
    
    # Get slide object
    slide = registrar.get_slide(slide_name)
    
    if slide is None:
        raise ValueError(f"Slide {slide_name} not found in registrar")
    
    # Transform coordinates
    transformed = []
    for x, y in coords:
        # VALIS uses (x, y) format
        warped_xy = slide.warp_xy(np.array([[x, y]]))[0]
        transformed.append(warped_xy)
    
    transformed = np.array(transformed)
    
    return transformed


def add_warped_coordinates(adata, warped_coords):
    """Add warped coordinates to AnnData."""
    adata.obsm['spatial_warped'] = warped_coords
    adata.obs['x_warped'] = warped_coords[:, 0]
    adata.obs['y_warped'] = warped_coords[:, 1]
    
    return adata

In [ ]:
# Transform coordinates for registered samples
if HAS_VALIS and 'registrar' in dir():
    print("Transforming coordinates...")
    
    # Get slide name (composite image)
    slide_name = f"{sample_id}_composite.tif"
    
    try:
        warped_coords = transform_coordinates(adata, registrar, slide_name)
        adata = add_warped_coordinates(adata, warped_coords)
        
        print(f"Original coords range: X[{adata.obsm['spatial'][:, 0].min():.0f}, {adata.obsm['spatial'][:, 0].max():.0f}], "
              f"Y[{adata.obsm['spatial'][:, 1].min():.0f}, {adata.obsm['spatial'][:, 1].max():.0f}]")
        print(f"Warped coords range: X[{warped_coords[:, 0].min():.0f}, {warped_coords[:, 0].max():.0f}], "
              f"Y[{warped_coords[:, 1].min():.0f}, {warped_coords[:, 1].max():.0f}]")
        
        # Save updated AnnData
        adata.write_h5ad(sample_file, compression='gzip')
        print(f"Saved warped coordinates to AnnData")
        
    except Exception as e:
        print(f"Error transforming coordinates: {e}")

## Visualization

In [ ]:
# Compare original and warped coordinates
if 'adata' in dir() and 'spatial_warped' in adata.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original
    ax = axes[0]
    scatter = ax.scatter(
        adata.obsm['spatial'][:, 0],
        adata.obsm['spatial'][:, 1],
        c=adata.obs['leiden'].astype(int) if 'leiden' in adata.obs else None,
        s=1, alpha=0.5, cmap='tab20'
    )
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title('Original Coordinates')
    ax.set_aspect('equal')
    
    # Warped
    ax = axes[1]
    scatter = ax.scatter(
        adata.obsm['spatial_warped'][:, 0],
        adata.obsm['spatial_warped'][:, 1],
        c=adata.obs['leiden'].astype(int) if 'leiden' in adata.obs else None,
        s=1, alpha=0.5, cmap='tab20'
    )
    ax.set_xlabel('X (warped)')
    ax.set_ylabel('Y (warped)')
    ax.set_title('Warped Coordinates')
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{sample_id}_coordinate_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

## Quality Assessment

In [ ]:
def compute_registration_quality(original_coords, warped_coords):
    """Compute registration quality metrics."""
    # Displacement statistics
    displacement = warped_coords - original_coords
    
    metrics = {
        'mean_displacement_x': np.mean(displacement[:, 0]),
        'mean_displacement_y': np.mean(displacement[:, 1]),
        'std_displacement_x': np.std(displacement[:, 0]),
        'std_displacement_y': np.std(displacement[:, 1]),
        'max_displacement': np.max(np.linalg.norm(displacement, axis=1)),
        'mean_displacement_magnitude': np.mean(np.linalg.norm(displacement, axis=1)),
    }
    
    return metrics

In [ ]:
if 'adata' in dir() and 'spatial_warped' in adata.obsm:
    quality_metrics = compute_registration_quality(
        adata.obsm['spatial'],
        adata.obsm['spatial_warped']
    )
    
    print("Registration Quality Metrics:")
    for k, v in quality_metrics.items():
        print(f"  {k}: {v:.2f}")
    
    # Save metrics
    metrics_df = pd.DataFrame([quality_metrics])
    metrics_df['sample_id'] = sample_id
    metrics_df.to_csv(OUTPUT_DIR / f"{sample_id}_registration_metrics.csv", index=False)

## Export Warped Coordinates

In [ ]:
if 'adata' in dir() and 'spatial_warped' in adata.obsm:
    # Export warped coordinates with cluster info
    export_df = pd.DataFrame({
        'x_original': adata.obsm['spatial'][:, 0],
        'y_original': adata.obsm['spatial'][:, 1],
        'x_warped': adata.obsm['spatial_warped'][:, 0],
        'y_warped': adata.obsm['spatial_warped'][:, 1],
    })
    
    if 'leiden' in adata.obs.columns:
        export_df['cluster'] = adata.obs['leiden'].values
    
    export_path = OUTPUT_DIR / f"{sample_id}_warped_coordinates.csv"
    export_df.to_csv(export_path, index=False)
    print(f"Exported warped coordinates to {export_path}")

## Summary

In [ ]:
print("\n" + "="*60)
print("REGISTRATION PIPELINE SUMMARY")
print("="*60)
print(f"\nVALIS available: {HAS_VALIS}")
print(f"Composite method: {COMPOSITE_METHOD}")
print(f"Samples configured: {len(SAMPLE_MAPPING)}")

# List output files
output_files = list(OUTPUT_DIR.glob("*"))
print(f"\nOutput files:")
for f in output_files[:20]:  # Limit to first 20
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  - {f.name}: {size_mb:.1f} MB")